In [ ]:
%reload_ext autoreload
%autoreload 2 
import sqlite3
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, quality, viz

from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

viz.apply_theme()

con = sqlite3.connect(config. DB_PATH)
DAY = config. ANALYSIS_DAY

NB = "D05_流水线与性能"

SOURCE = f"{config.SOURCE_NOTE};脚本:{NB};制图:{config.AUTHOR}"

pd.set_option("display.max_columns", 20)

In [ ]:
from signal_insight import dwd

raw = pd.read_sql("SELECT * FROM ods_signal", con)
dwd_new, steps = dwd.clean_signal(raw, DAY)
dwd_d02 = pd.read_sql("SELECT * FROM dwd_signal WHERE dt = ?",con,params=(DAY,),parse_dates=['ts'])

pd.DataFrame(steps)

In [ ]:
# 按日期分区写两遍:结果不变(幂等);顺便用正确的时间格式把D02 丢掉的毫秒补回来

from signal_insight import db
for _ in range(2):
    db.write_partition(con, dwd_new, "dwd_signal", DAY)

n = pd.read_sql("SELECT COUNT(*) AS n FROM dwd_signal WHERE dt = ?",con,params=(DAY, ))["n"][0]

back = pd.read_sql("SELECT ts FROM dwd_signal WHERE dt = ?",con,params=(DAY, ),parse_dates=["ts"])

print(f"写了两遍,{DAY} 仍是{n:,}行; 读回来的时间空值 {back['ts'].isna().sum()}个; \
带毫秒的行 {(back['ts'].dt.microsecond > 0).sum():,}")

In [ ]:
from signal_insight import dims

districts = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")

aois = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_aoi_v2", con), "name"
)

dim_new = dims.build_dim_cell(
    pd.read_sql("SELECT * FROM ods_cell", con), districts, aois)
result = pd.testing.assert_frame_equal(dim_new,
    pd.read_sql("SELECT * FROM dim_cell", con), check_dtype=False
)

hot_new = dims.build_dim_hotspot(dwd_new, dim_new)
hot_d04 = pd.read_sql("SELECT * FROM dim_hotspot", con)
hot_d04  # 正常来说hot_d04和hot_new是一致的

In [ ]:
from signal_insight import pipeline

log = pipeline.run_day() # 执行流水线流程
chk = pd.read_sql(
    "SELECT (SELECT COUNT(*) FROM dwd_signal WHERE dt = '2018-10-03') AS dwd, "
    "(SELECT COUNT(*) FROM dim_cell) AS dim_cell, (SELECT name FROM dim_hotspot) AS hotspot, "
    "(SELECT COUNT(DISTINCT 批次) FROM etl_log) AS runs", con)
etl_log = log[["步骤", "输入行数", "输出行数", "耗时(秒)"]]

In [ ]:
etl_log

In [ ]:
# D02 登记稽核时，有两条编号写成了下划线（DQ_02、DQ_03）。排序时 "_" 排在 "-" 后面，这两条会跑到 DQ-16 之后。
# 统一成短横线。D02 notebook 里那两处 dq_record 也要同步改成 DQ-02、DQ-03，否则重跑 D02 又会写回来。
con.execute("UPDATE dq_check_result SET check_id = REPLACE(check_id, '_', '-') WHERE check_id IN ('DQ_02', 'DQ_03')")
con.commit()
ids = pd.read_sql("SELECT check_id FROM dq_check_result", con)["check_id"]
assert ids.is_unique and ids.str.fullmatch(r"DQ-\d{2}").all(), ids[~ids.str.fullmatch(r"DQ-\d{2}")].tolist()
print(f"✅ 稽核表 {len(ids)} 条，编号唯一、格式统一（DQ-01 … DQ-{ids.str[3:].astype(int).max():02d}）")

In [ ]:
dq = pd.read_sql("SELECT * FROM dq_check_result", con).sort_values("check_id")
summary = (dq.pivot_table(index="维度", columns="状态", values="check_id",
         aggfunc="count").reindex(columns=["不通过", "警告", "通过"], fill_value=0))

summary["问题数"] = summary["不通过"] + summary["警告"]
summary = summary.sort_values(["问题数", "不通过"])
summary.to_sql("ads_dq_summary", con, if_exists="replace")
dq[["check_id", "维度", "检查项", "结果", "状态"]]